# Sesión 10 A



## 2.2. Patrones de razonamiento e inferencia

Teniendo una situación modelada con una red Bayesiana, nos podemos plantear **tres** tipos básicos de razonamiento de podríamos querer resolver:

* Razonamiento causal
* Razonamiento evidencial
* Razonamiento intercausal

**1. Razonamiento causal**

El **razonamiento causal** sigue la dirección natural de las flechas del grafo: va de **causa → efecto**, o de **nodo padre → nodo hijo**.

> Si sé algo sobre las causas, ¿qué puedo inferir sobre sus efectos?

![causal-reasoning](../images/sesion10-student-model-causal.png)

Por ejemplo, 

**Pregunta**: ¿cuál es la probabilidad de obtener una buena carta de recomendación?

$$P(r^1) = \sum_{D,I,C,E} P(D,I,C,E,r^1) \approx ?$$

In [1]:
from pgmpy.models import BayesianNetwork, DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

In [2]:
student_model = DiscreteBayesianNetwork(
    [("D", "C"), ("I", "C"), ("I", "E"), ("C", "R")]
)

# CPDs
cpd_D = TabularCPD(
    variable='D',
    variable_card=2,
    values=[
        [0.6],
        [0.4]
    ]
)
cpd_I = TabularCPD(
    variable='I',
    variable_card=2,
    values=[
        [0.7],
        [0.3]
    ]
)

cpd_C = TabularCPD(
    variable='C',
    variable_card=3,
    values=[
        [0.30, 0.70, 0.02, 0.20],
        [0.40, 0.25, 0.08, 0.30],
        [0.30, 0.05, 0.90, 0.50]
    ],
    evidence=['I', 'D'], 
    evidence_card=[2, 2] 
)
cpd_E = TabularCPD(
    variable='E',
    variable_card=2,
    values=[
        [0.95, 0.20],
        [0.05, 0.80]
    ],
    evidence=['I'],
    evidence_card=[2]
)
cpd_R = TabularCPD(
    variable='R',
    variable_card=2,
    values=[
        [0.99, 0.40, 0.10],
        [0.01, 0.60, 0.90]
    ],
    evidence=['C'],
    evidence_card=[3]
)

In [3]:
student_model.add_cpds(cpd_D, cpd_I, cpd_C, cpd_E, cpd_R)

In [5]:
# Obtenemos la distribución conjunta de la red

p_joint = (
    cpd_I.to_factor()
    * cpd_D.to_factor()
    * cpd_C.to_factor()
    * cpd_E.to_factor()
    * cpd_R.to_factor()
)
print(p_joint)

+------+------+------+------+------+------------------+
| I    | D    | C    | E    | R    |   phi(I,D,C,E,R) |
+======+======+======+======+======+==================+
| I(0) | D(0) | C(0) | E(0) | R(0) |           0.1185 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(0) | E(0) | R(1) |           0.0012 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(0) | E(1) | R(0) |           0.0062 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(0) | E(1) | R(1) |           0.0001 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(1) | E(0) | R(0) |           0.0638 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(1) | E(0) | R(1) |           0.0958 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(1) | E(1) | R(0) |           0.0034 |
+------+------+------+------+------+------------------+
| I(0) | D(0) | C(1) | E(1) | R(1) |           0

In [5]:
#print

In [6]:
# Marginalizar sobre las variables I, D, C, E
p_R = p_joint.marginalize(variables=['I', 'D', 'C', 'E'], inplace=False)
print(p_R)

+------+----------+
| R    |   phi(R) |
+======+==========+
| R(0) |   0.4977 |
+------+----------+
| R(1) |   0.5023 |
+------+----------+


In [8]:
#reduce
p_r1 = p_R.reduce(values=[('R', 1)], inplace=False)
print(p_r1)

+---------+
|   phi() |
+=========+
|  0.5023 |
+---------+


Sin embargo, podemos evaluar cómo esta probabilidad cambia si la condicionamos sobre la inteligencia. Por ejemplo, si el estudiante no es muy inteligente

$$P(r^1 | i^0) = \frac{P(r^1, i^0)}{P(i^0)} = \frac{\sum_{D,C,E} P(D, i^0, C, E, r^1)}{\sum_{D,C,E,R} P(D, i^0, C, E, R)} \approx ?$$

In [9]:
# marginalizar sobre las variables D, C, E // numerador
p_RI = p_joint.marginalize(variables=['D', 'C', 'E'], inplace=False)
print(p_RI)

+------+------+------------+
| I    | R    |   phi(I,R) |
+======+======+============+
| I(0) | R(0) |     0.4280 |
+------+------+------------+
| I(0) | R(1) |     0.2720 |
+------+------+------------+
| I(1) | R(0) |     0.0697 |
+------+------+------------+
| I(1) | R(1) |     0.2303 |
+------+------+------------+


In [10]:
# get_value
i_0 = p_RI.get_value(R=1, I=0)
i_0

np.float64(0.27202000000000004)

In [11]:
# marginalizar sobre las variables D, C, E, R // denominador
p_I = p_joint.marginalize(variables=['D', 'C', 'E', 'R'], inplace=False)
print(p_I)

+------+----------+
| I    |   phi(I) |
+======+==========+
| I(0) |   0.7000 |
+------+----------+
| I(1) |   0.3000 |
+------+----------+


In [12]:
# get_value
r1_i0 = p_I.get_value(I=0)
r1_i0

np.float64(0.7)

**¿se esperaba esto o no?**

In [13]:
# print probabilidad de r1 dado i0
i_0/r1_i0

np.float64(0.38860000000000006)

Por otra parte, si también condicionamos sobre la dificultad

$$P(r^1 | i^0, d^0) = \frac{P(r^1, i^0, d^0)}{P(i^0, d^0)} = \frac{\sum_{C,P} P(d^0, i^0, C, E, r^1)}{\sum_{C,E,R} P(d^0, i^0, C, E, R)} \approx ?$$

In [14]:
# Marginalizar sobre las variables C, E, R // numerador
p_RID = p_joint.marginalize(variables=['C','E'], inplace=False)
r1_i0_d0 = p_RID.get_value(R=1, I=0, D=0)
r1_i0_d0

np.float64(0.21546)

In [15]:
# Marginalizar sobre las variables C, E, R, I // denominador
p_ID = p_joint.marginalize(variables=['C', 'E', 'R'], inplace=False)
i0_d0 = p_ID.get_value(I=0, D=0)
i0_d0

np.float64(0.41999999999999993)

**¿Se esperaba esto o no?**

In [16]:
#print probabilidad de r1 dado i0 y d0
r1_i0_d0 / i0_d0

np.float64(0.5130000000000001)

---

**2. Razonamiento evidencial**

Va **de efecto a causa**, en sentido contrario a las flechas.

> Si observo un efecto, ¿qué puedo inferir sobre sus causas?

![causal-reasoning](../images/sesion10-student-model-evid.png)

Por ejemplo, la probabilidad de que el curso sea difícil es:

$$P(d^1) = 0.4$$

Condicionando sobre la calificación:

$$P(d^1 | c^0) = \frac{P(d^1, c^0)}{P(c^0)} = \frac{\sum_{I,E,R} P(d^1, I, c^0, E, R)}{\sum_{D,I,E,R} P(D, I, c^0, E, R)} \approx?$$

In [17]:
# marginalizar sobre las variables I, E, R // numerador
p_DC = p_joint.marginalize(
    variables=['I', 'E', 'R'],
    inplace=False
)
p_d1_c0 = p_DC.get_value(D=1, C=0)
p_d1_c0

np.float64(0.21999999999999995)

In [19]:
# marginalizar sobre las variables D, I, E, R // denominador
p_C = p_joint.marginalize(
    variables=['D', 'I', 'E', 'R'],
    inplace=False
)
p_c0 = p_C.get_value(C=0)
p_c0

np.float64(0.34959999999999997)

In [20]:
# print probabilidad de d1 dado c0
p_d1_c0/p_c0

np.float64(0.6292906178489701)

In [21]:
#Otra forma de calcular P(D1 | C0) usando inferencia en la red bayesiana    
from pgmpy.inference import VariableElimination

infer = VariableElimination(student_model)

phi = infer.query(
    variables=['D'],
    evidence={'C': 0}
)
phi.values[1]

np.float64(0.6292906178489702)

> Intuición: observar una calificación baja hace más probable que el curso haya sido difícil (sube de $0.4$ a $\approx 0.63$).

---

Similarmente, la probabilidad de que el estudiante sea inteligente es:

$$P(i^1) = 0.3$$

Condicionando sobre la calificación:

$$P(i^1 | c^0) = \frac{P(i^1, c^0)}{P(c^0)} = \frac{\sum_{D,E,R} P(D, i^1, c^0, E, R)}{\sum_{D,I,E,R} P(D, I, c^0, E, R)} \approx ?$$

In [ ]:
# marginalizar sobre las variables D, E, R // numerador
p_IC = p_joint.marginalize(
    variables=['D', 'E', 'R'],
    inplace=False
)
p_i1_c0 = p_IC.get_value(I=1, C=0)

# marginalizar sobre las variables D, I, E, R // denominador
p_C = p_joint.marginalize(
    variables=['D', 'I', 'E', 'R'],
    inplace=False
)

p_c0 = p_C.get_value(C=0)

In [24]:
# print probabilidad de i1 dado c0
p_i1_c0/p_c0

np.float64(0.07894736842105264)

In [25]:
# o con VariableElimination
infer = VariableElimination(student_model)

phi = infer.query(
    variables=['I'],
    evidence={'C': 0}
)
phi.values[1]

np.float64(0.07894736842105264)

> Intuición: observar una calificación baja hace menos probable que el estudiante sea inteligente (baja del $0.3$ a $\approx 0.11$).

**3. Razonamiento intercausal**

Ocurre cuando **dos causas comparten un mismo efecto** y una de ellas se observa.

> Si conozco una causa, ¿cómo cambia mi creencia sobre la otra, dado que comparten el mismo efecto?


![intercausal-reasoning](../images/sesion10-student-model-inter.png )

$\text{Dificultad} \longrightarrow \text{Calificación} \longleftarrow \text{Inteligencia}$

Normalmente, $D$ e $I$ son independientes. Pero, una vez que conocemos el efecto común -por ejemplo, la calificación $C$-, dejan de serlo.

> Si sabemos que la calificación fue alta y que el curso era difícil, es más probable que el estudiante haya sido inteligente.

Antes de observar $C$:

$$ D \perp I $$

Después de observar $C$:
$$ D \not\perp I \mid C $$

De nuevo, la probabilidad de que el estudiante sea inteligente es:

$$P(i^1) = 0.3$$

Condicionando sobre la calificación:

$$P(i^1 | c^0) = \frac{P(i^1, c^0)}{P(c^0)} \approx 0.07$$

Aún más, si condicionamos sobre la dificultad:

$$P(i^1 | c^0, d^1) = \frac{P(i^1, c^0, d^1)}{P(c^0, d^1)} \approx ?$$

In [22]:
# VariableElimination
infer = VariableElimination(student_model)
phi = infer.query(
    variables=['I'],
    evidence={'C':0, 'D':1}
)
phi.values[1]

np.float64(0.1090909090909091)

> Inicialmente, el estudiante tiene una probabilidad moderada de ser inteligente ($P(i^1)=0.3$). Al observar que obtuvo una **mala calificación**, esa creencia **disminuye drásticamente** ($P(i^1 \mid c^0) \approx 0.07$). Sin embargo, si además sabemos que el curso era **difícil**, parte de la mala nota se explica por la dificultad, por lo que la probabilidad de que sea inteligente **vuelve a subir ligeramente** $P(i^1 \mid c^0, d^1) \approx 0.11$.

In [24]:
#guardar el modelo
#import pickle

#with open('student-model.pkl', 'wb') as f:
#    pickle.dump(student_model, f)